In [ ]:
"""import sys
from PySide6.QtWidgets import QApplication, QMainWindow

class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()

        # Set up the main window properties
        self.setWindowTitle("Spatial Processing Tool")
        self.setGeometry(100, 100, 800, 600) # (x, y, width, height)

        # Initialize the menu bar
        self.create_menu_bar()

    def create_menu_bar(self):
        # Create the main menu bar attached to the window
        menu_bar = self.menuBar()

        # The list of menus you requested
        menu_categories = [
            "&File", "&Basic", "&Tools", "&Classification", "&Transform",
            "F&ilter", "&Spectral", "&Map", "&Vector", "T&opographic",
            "&Radar", "&Window", "&Help"
        ]

        # Loop through the list to generate the top-level menus
        for category in menu_categories:
            menu = menu_bar.addMenu(category)
            
            # --- Adding Drop-Down Options (Actions) ---
            # We add a couple of placeholder actions to demonstrate the drop-down
            clean_name = category.replace("&", "") # Removing the ampersand for the action name
            
            if clean_name == "File":
                # The File menu gets specific examples
                open_action = menu.addAction("Open...")
                save_action = menu.addAction("Save")
                menu.addSeparator() # Adds a visual divider line
                exit_action = menu.addAction("Exit")
                
                # Connect the exit action to actually close the app
                exit_action.triggered.connect(self.close)
            else:
                # Generic placeholders for the rest of the menus
                menu.addAction(f"{clean_name} Option 1")
                menu.addAction(f"{clean_name} Option 2")

if __name__ == "__main__":
    # Every PySide6 application requires exactly one QApplication instance
    app = QApplication(sys.argv)
    
    # Create and display the window
    window = MainWindow()
    window.show()
    
    # Start the event loop
    sys.exit(app.exec())"""

In [ ]:
"""class shape(self, lenght, breadth):
    lengths=2
    dog=lenght

x=shape(3)
print(x.length)"""

In [ ]:
'''import sys
import numpy as np
import rasterio
from PySide6.QtWidgets import (QApplication, QMainWindow, QWidget, QHBoxLayout, 
                               QVBoxLayout, QFormLayout, QRadioButton, QSpinBox, 
                               QPushButton, QGroupBox, QMessageBox, QLabel)
from PySide6.QtCore import Qt

# Imports for embedding Matplotlib in PySide6
from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure

class TifViewerWindow(QMainWindow):
    """A standalone window with side-panel controls for hyperspectral .tif rendering."""
    
    def __init__(self, filepath):
        super().__init__()
        self.filepath = filepath
        self.setWindowTitle(f"Hyperspectral Viewer - {filepath}")
        self.resize(1000, 700)
        
        # 1. Fetch file info (total bands) to configure the UI limits
        try:
            with rasterio.open(filepath) as src:
                self.max_bands = src.count
        except Exception as e:
            QMessageBox.critical(self, "Error", f"Failed to open file: {e}")
            self.max_bands = 1 # Fallback to prevent crashes

        # 2. Main Layout Setup (Horizontal Split)
        central_widget = QWidget()
        self.setCentralWidget(central_widget)
        main_layout = QHBoxLayout(central_widget)
        
        # -- Left Side: Matplotlib Canvas --
        self.canvas = FigureCanvas(Figure(figsize=(8, 6)))
        self.ax = self.canvas.figure.subplots()
        self.ax.axis('off')
        main_layout.addWidget(self.canvas, stretch=3) # Takes 3/4 of the space
        
        # -- Right Side: Control Panel --
        controls_layout = QVBoxLayout()
        main_layout.addLayout(controls_layout, stretch=1) # Takes 1/4 of the space
        
        self.setup_control_panel(controls_layout)
        
        # 3. Initial Load
        self.render_image()

    def setup_control_panel(self, layout):
        """Builds the sidebar with mode selection and band selectors."""
        
        # File Info Label
        info_label = QLabel(f"<b>Total Bands Available:</b> {self.max_bands}")
        layout.addWidget(info_label)
        
        # --- Mode Selection Group ---
        mode_group = QGroupBox("Visualization Mode")
        mode_layout = QVBoxLayout()
        
        self.radio_rgb = QRadioButton("RGB Composite")
        self.radio_rgb.setChecked(True)
        self.radio_rgb.toggled.connect(self.toggle_ui_mode)
        
        self.radio_gray = QRadioButton("Grayscale (Single Band)")
        
        mode_layout.addWidget(self.radio_rgb)
        mode_layout.addWidget(self.radio_gray)
        mode_group.setLayout(mode_layout)
        layout.addWidget(mode_group)
        
        # --- Band Selectors Group ---
        bands_group = QGroupBox("Band Selection")
        bands_layout = QFormLayout()
        
        # Helper to create bounded spinboxes
        def create_spinbox(default_val):
            sb = QSpinBox()
            sb.setMinimum(1)
            sb.setMaximum(self.max_bands)
            sb.setValue(min(default_val, self.max_bands)) # Prevent setting out-of-bounds default
            return sb

        self.spin_r = create_spinbox(45)
        self.spin_g = create_spinbox(29)
        self.spin_b = create_spinbox(15)
        
        # We will use the 'Red' spinbox as the single band selector for Grayscale
        self.label_r = QLabel("Red Band:")
        self.label_g = QLabel("Green Band:")
        self.label_b = QLabel("Blue Band:")
        
        bands_layout.addRow(self.label_r, self.spin_r)
        bands_layout.addRow(self.label_g, self.spin_g)
        bands_layout.addRow(self.label_b, self.spin_b)
        
        bands_group.setLayout(bands_layout)
        layout.addWidget(bands_group)
        
        # --- Update Button ---
        self.btn_update = QPushButton("Update Image")
        self.btn_update.setMinimumHeight(40)
        self.btn_update.clicked.connect(self.render_image)
        layout.addWidget(self.btn_update)
        
        # Push everything to the top
        layout.addStretch()

    def toggle_ui_mode(self):
        """Enables/Disables Green and Blue inputs based on the selected mode."""
        is_rgb = self.radio_rgb.isChecked()
        
        # Toggle visibility/enablement
        self.spin_g.setVisible(is_rgb)
        self.spin_b.setVisible(is_rgb)
        self.label_g.setVisible(is_rgb)
        self.label_b.setVisible(is_rgb)
        
        # Rename the first spinbox based on context
        if is_rgb:
            self.label_r.setText("Red Band:")
        else:
            self.label_r.setText("Gray Band:")

    def render_image(self):
        """Reads the selected bands, normalizes the data, and plots it."""
        try:
            with rasterio.open(self.filepath) as src:
                vmin_val = 110.0
                vmax_val = 2000.0
                
                self.ax.clear() # Clear the previous plot
                self.ax.axis('off')
                
                if self.radio_rgb.isChecked():
                    # --- RGB Mode ---
                    r = src.read(self.spin_r.value())
                    g = src.read(self.spin_g.value())
                    b = src.read(self.spin_b.value())
                    
                    img = np.dstack((r, g, b))
                    
                    # Normalize
                    img = np.clip(img, vmin_val, vmax_val)
                    img = (img - vmin_val) / (vmax_val - vmin_val)
                    
                    self.ax.imshow(img)
                    
                else:
                    # --- Grayscale Mode ---
                    gray = src.read(self.spin_r.value())
                    
                    # Normalize
                    gray = np.clip(gray, vmin_val, vmax_val)
                    gray = (gray - vmin_val) / (vmax_val - vmin_val)
                    
                    # Plot using a grayscale colormap
                    self.ax.imshow(gray, cmap='gray')
                
                self.canvas.draw()
                
        except Exception as e:
            QMessageBox.critical(self, "Rendering Error", f"Could not load bands:\n{e}")


# ==========================================
# Example Usage
# ==========================================
if __name__ == "__main__":
    app = QApplication(sys.argv)
    
    # Pass a valid path for testing if running standalone
    target_file_path = "sample.tif" if len(sys.argv) <= 1 else sys.argv
        
    viewer = TifViewerWindow(target_file_path)
    viewer.show()
    
    sys.exit(app.exec())'''

In [ ]:
"""import sys
from PySide6.QtWidgets import (QApplication, QMainWindow, QPushButton, 
                                QLabel, QVBoxLayout, QWidget, QFileDialog, 
                                QMessageBox, QScrollArea)
from PySide6.QtGui import QPixmap, QImage
from PySide6.QtCore import Qt  # ← ADD THIS IMPORT
from PIL import Image
import numpy as np


class TIFImageViewer(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("TIFF Image Viewer")
        self.setGeometry(100, 100, 800, 600)
        
        # Create central widget and layout
        central_widget = QWidget()
        self.setCentralWidget(central_widget)
        layout = QVBoxLayout(central_widget)
        
        # Create button to open image
        self.open_button = QPushButton("Open TIFF Image")
        self.open_button.clicked.connect(self.open_image)
        layout.addWidget(self.open_button)
        
        # Create scroll area for image display
        self.scroll_area = QScrollArea()
        self.scroll_area.setWidgetResizable(True)
        layout.addWidget(self.scroll_area)
        
        # Create label for image display
        self.image_label = QLabel("No image loaded")
        self.image_label.setAlignment(Qt.AlignCenter)  # Now works!
        self.scroll_area.setWidget(self.image_label)
        
        self.current_image = None
    
    def open_image(self):
        # Open file dialog
        file_path, _ = QFileDialog.getOpenFileName(
            self,
            "Open TIFF Image",
            "",
            "TIFF Files (*.tif *.tiff);;All Files (*)"
        )
        
        if file_path:
            try:
                # Open TIFF image using Pillow
                pil_image = Image.open(file_path)
                
                # Convert to RGB if necessary (for multi-channel TIFFs)
                if pil_image.mode != 'RGB':
                    pil_image = pil_image.convert('RGB')
                
                # Convert PIL Image to QImage
                image_data = pil_image.tobytes("raw", "RGB")
                qimage = QImage(
                    image_data,
                    pil_image.width,
                    pil_image.height,
                    pil_image.width * 3,
                    QImage.Format_RGB888
                )
                
                # Convert to QPixmap and display
                pixmap = QPixmap.fromImage(qimage)
                self.image_label.setPixmap(pixmap)
                self.image_label.setText("")
                self.current_image = pil_image
                
                # Resize window to fit image (optional)
                self.adjustSize()
                
            except Exception as e:
                QMessageBox.critical(
                    self,
                    "Error",
                    f"Failed to open image:\n{str(e)}"
                )


if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = TIFImageViewer()
    window.show()
    sys.exit(app.exec())
"""

In [ ]:
'''import sys
import numpy as np
from PySide6.QtWidgets import (QApplication, QMainWindow, QPushButton, 
                                QLabel, QVBoxLayout, QWidget, QFileDialog, 
                                QMessageBox, QScrollArea, QHBoxLayout, 
                                QSpinBox, QFormLayout, QGroupBox)
from PySide6.QtGui import QPixmap, QImage
from PySide6.QtCore import Qt
from PIL import Image
import tifffile


class HyperspectralViewer(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Hyperspectral TIFF Viewer")
        self.setGeometry(100, 100, 1000, 700)
        
        # Create central widget and layout
        central_widget = QWidget()
        self.setCentralWidget(central_widget)
        main_layout = QHBoxLayout(central_widget)
        
        # Left panel - Controls
        control_panel = QWidget()
        control_layout = QVBoxLayout(control_panel)
        control_panel.setFixedWidth(250)
        
        # Open button
        self.open_button = QPushButton("Open Hyperspectral TIFF")
        self.open_button.clicked.connect(self.open_image)
        control_layout.addWidget(self.open_button)
        
        # Band selection group
        band_group = QGroupBox("Band Selection (RGB)")
        band_layout = QFormLayout()
        
        self.red_band = QSpinBox()
        self.red_band.setRange(0, 1000)
        self.red_band.setValue(0)
        self.red_band.valueChanged.connect(self.update_display)
        
        self.green_band = QSpinBox()
        self.green_band.setRange(0, 1000)
        self.green_band.setValue(1)
        self.green_band.valueChanged.connect(self.update_display)
        
        self.blue_band = QSpinBox()
        self.blue_band.setRange(0, 1000)
        self.blue_band.setValue(2)
        self.blue_band.valueChanged.connect(self.update_display)
        
        band_layout.addRow("Red Band:", self.red_band)
        band_layout.addRow("Green Band:", self.green_band)
        band_layout.addRow("Blue Band:", self.blue_band)
        band_group.setLayout(band_layout)
        control_layout.addWidget(band_group)
        
        # Image info
        self.info_group = QGroupBox("Image Info")
        info_layout = QFormLayout()
        self.info_label = QLabel("No image loaded")
        self.info_label.setWordWrap(True)
        info_layout.addRow(self.info_label)
        self.info_group.setLayout(info_layout)
        control_layout.addWidget(self.info_group)
        
        control_layout.addStretch()
        main_layout.addWidget(control_panel)
        
        # Right panel - Image display
        self.scroll_area = QScrollArea()
        self.scroll_area.setWidgetResizable(True)
        main_layout.addWidget(self.scroll_area)
        
        self.image_label = QLabel("No image loaded")
        self.image_label.setAlignment(Qt.AlignCenter)
        self.scroll_area.setWidget(self.image_label)
        
        self.hyperspectral_data = None
        self.num_bands = 0
    
    def open_image(self):
        file_path, _ = QFileDialog.getOpenFileName(
            self,
            "Open Hyperspectral TIFF",
            "",
            "TIFF Files (*.tif *.tiff);;All Files (*)"
        )
        
        if file_path:
            try:
                # Load hyperspectral image using tifffile
                self.hyperspectral_data = tifffile.imread(file_path)
                
                # Handle different data shapes
                if len(self.hyperspectral_data.shape) == 3:
                    # Shape: (bands, height, width) or (height, width, bands)
                    if self.hyperspectral_data.shape[0] < self.hyperspectral_data.shape[1]:
                        # Assume (bands, height, width)
                        self.hyperspectral_data = np.transpose(
                            self.hyperspectral_data, (1, 2, 0)
                        )
                    self.num_bands = self.hyperspectral_data.shape[2]
                elif len(self.hyperspectral_data.shape) == 2:
                    # Single band grayscale
                    self.hyperspectral_data = self.hyperspectral_data[:, :, np.newaxis]
                    self.num_bands = 1
                else:
                    raise ValueError("Unsupported image shape")
                
                # Update band spinners
                self.red_band.setMaximum(self.num_bands - 1)
                self.green_band.setMaximum(self.num_bands - 1)
                self.blue_band.setMaximum(self.num_bands - 1)
                
                # Set default bands (try to find good RGB approximation)
                if self.num_bands >= 3:
                    # Common hyperspectral band assignments
                    self.red_band.setValue(min(30, self.num_bands - 1))
                    self.green_band.setValue(min(20, self.num_bands - 1))
                    self.blue_band.setValue(min(10, self.num_bands - 1))
                
                # Update info
                info_text = (
                    f"Shape: {self.hyperspectral_data.shape}\n"
                    f"Bands: {self.num_bands}\n"
                    f"Dtype: {self.hyperspectral_data.dtype}\n"
                    f"Min: {self.hyperspectral_data.min()}\n"
                    f"Max: {self.hyperspectral_data.max()}"
                )
                self.info_label.setText(info_text)
                
                # Display initial image
                self.update_display()
                
            except Exception as e:
                QMessageBox.critical(
                    self,
                    "Error",
                    f"Failed to open image:\n{str(e)}"
                )
    
    def update_display(self):
        if self.hyperspectral_data is None:
            return
        
        try:
            # Get selected bands
            r_band = self.red_band.value()
            g_band = self.green_band.value()
            b_band = self.blue_band.value()
            
            # Ensure bands are within range
            r_band = min(r_band, self.num_bands - 1)
            g_band = min(g_band, self.num_bands - 1)
            b_band = min(b_band, self.num_bands - 1)
            
            # Extract RGB bands
            red = self.hyperspectral_data[:, :, r_band]
            green = self.hyperspectral_data[:, :, g_band]
            blue = self.hyperspectral_data[:, :, b_band]
            
            # Stack into RGB
            rgb_image = np.stack([red, green, blue], axis=2)
            
            # Normalize to 0-255
            rgb_image = self.normalize_to_8bit(rgb_image)
            
            # Convert to QImage
            height, width, channels = rgb_image.shape
            bytes_per_line = channels * width
            qimage = QImage(
                rgb_image.tobytes(),
                width,
                height,
                bytes_per_line,
                QImage.Format_RGB888
            )
            
            # Display
            pixmap = QPixmap.fromImage(qimage)
            self.image_label.setPixmap(pixmap)
            
        except Exception as e:
            QMessageBox.warning(
                self,
                "Display Error",
                f"Failed to display image:\n{str(e)}"
            )
    
    def normalize_to_8bit(self, image):
        """Normalize image data to 8-bit RGB"""
        img_min = image.min()
        img_max = image.max()
        
        if img_max > img_min:
            # Normalize to 0-255
            image = (image - img_min) / (img_max - img_min) * 255.0
        else:
            image = np.zeros_like(image)
        
        return image.astype(np.uint8)


if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = HyperspectralViewer()
    window.show()
    sys.exit(app.exec())
'''

In [ ]:
class Player:
    def __init__(self, name, level):
        self.name = name
        self.level = level

    def introduce(hen):
        # 'self' refers to this instance of the class
        print(f"Player: {hen.name}, Level: {hen.level}")

    def play(crow,var):
        # Increment the attribute for this specific instance
        print(var,crow)
        var.level += 1

player1 = Player("Alice", 1)
player2 = Player("Bob", 5)

player1.introduce()  # 'self' becomes player1
player1.play(player2)
player1
player2.level

Player: Alice, Level: 1
<__main__.Player object at 0x0000021B79A00910> <__main__.Player object at 0x0000021B7999A120>
